# 🧪 PRDetect — So sánh GCN2 vs GATv2 trên HC3 & RAID

**Mục tiêu:** Chứng minh GATv2 (chú ý đồ thị động) chống nhiễu đối kháng tốt hơn GCN2 (tĩnh)

**Các bước:**
1. Cài đặt thư viện
2. Tải dataset HC3 + RAID
3. Xây dựng đồ thị cú pháp (Dependency Syntax Tree)
4. Định nghĩa model GCN2 (baseline) + GATv2 (cải tiến)
5. Huấn luyện cả hai trên HC3
6. So sánh kết quả trên HC3 test + RAID test

> ⚠️ **Yêu cầu:** Chọn **GPU T4** hoặc **P100** trong Kaggle Settings trước khi chạy!

## Bước 1: Cài đặt thư viện

In [ ]:
# Cài PyTorch Geometric và các dependency (KHÔNG cài raid-bench để tránh xung đột numpy ABI)
!pip install torch-geometric -q
!pip install transformers -q
!pip install datasets -q
!python -m spacy download en_core_web_sm -q

print("✅ Cài đặt hoàn tất!")


In [ ]:
# Kiểm tra môi trường
import torch
import torch_geometric
print(f"PyTorch: {torch.__version__}")
print(f"PyG: {torch_geometric.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Bước 2: Tải Dataset HC3

In [ ]:
from datasets import load_dataset
import json
import os
import random

# Tải HC3 từ HuggingFace
print('Đang tải HC3 dataset từ HuggingFace...')
try:
    hc3 = load_dataset('Hello-SimpleAI/HC3', 'all', trust_remote_code=True)
except Exception as e:
    print('Thử tải trực tiếp file all.jsonl...')
    hc3 = load_dataset('json', data_files='https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/main/all.jsonl')

print(f'HC3 loaded! Keys: {list(hc3.keys())}')
print(f'Train samples: {len(hc3["train"])}')


In [ ]:
# Xem cấu trúc dữ liệu
sample = hc3['train'][0]
print("Các trường dữ liệu:", list(sample.keys()))
print(f"\nQuestion: {sample['question'][:100]}...")
print(f"Human answer (số lượng): {len(sample['human_answers'])}")
print(f"ChatGPT answer (số lượng): {len(sample['chatgpt_answers'])}")

In [ ]:
def convert_hc3_to_prdetect(hc3_data, max_samples=None):
    """
    Chuyển đổi HC3 sang format PRDetect: list of {text, label}
    """
    texts = []
    labels = []
    
    for item in hc3_data:
        for ans in item['human_answers']:
            if ans and len(ans.strip()) > 30:
                texts.append(ans.strip())
                labels.append(1)  # human = 1
        for ans in item['chatgpt_answers']:
            if ans and len(ans.strip()) > 30:
                texts.append(ans.strip())
                labels.append(0)  # machine = 0
    
    combined = list(zip(texts, labels))
    random.seed(2024)
    random.shuffle(combined)
    
    if max_samples:
        combined = combined[:max_samples]
    
    texts, labels = zip(*combined)
    return list(texts), list(labels)

# Chuyển đổi — lấy 4000 mẫu để train trên Kaggle
# (Tăng lên 8000-10000 nếu muốn kết quả tốt hơn, nhưng sẽ lâu hơn)
all_texts, all_labels = convert_hc3_to_prdetect(hc3['train'], max_samples=4000)

# Chia train/val/test = 70/15/15
n = len(all_texts)
n_train = int(n * 0.7)
n_val = int(n * 0.85)

train_texts, train_labels = all_texts[:n_train], all_labels[:n_train]
val_texts, val_labels = all_texts[n_train:n_val], all_labels[n_train:n_val]
test_texts, test_labels = all_texts[n_val:], all_labels[n_val:]

human_train = sum(1 for l in train_labels if l == 1)
machine_train = sum(1 for l in train_labels if l == 0)

print(f"Train: {len(train_texts)} mẫu (human={human_train}, machine={machine_train})")
print(f"Val:   {len(val_texts)} mẫu")
print(f"Test:  {len(test_texts)} mẫu")

## Bước 3: Xây dựng Đồ thị Cú pháp

In [ ]:
import spacy
from transformers import RobertaTokenizer, RobertaModel
from tqdm.notebook import tqdm
import time

print("Đang nạp SpaCy + RoBERTa...")
nlp = spacy.load("en_core_web_sm")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
roberta_model = RobertaModel.from_pretrained("roberta-base").to(device)
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base", do_lower_case=False)
roberta_model.eval()
print(f"✅ Models loaded on {device}")

In [ ]:
def build_graph_data(texts, labels, desc="Building graphs"):
    """Xây dựng đồ thị cú pháp cho danh sách văn bản."""
    all_token_embeddings = []
    all_edge_index = []
    valid_labels = []
    errors = 0
    
    for idx, text in enumerate(tqdm(texts, desc=desc)):
        try:
            doc = nlp(text)
            tokenized_sentence = [token.text for token in doc]
            if len(tokenized_sentence) < 3:
                errors += 1
                continue
            
            max_length = 512
            chunks = [tokenized_sentence[i:i+max_length] 
                      for i in range(0, len(tokenized_sentence), max_length)]
            chunk_outputs = []
            for chunk in chunks:
                token_ids = roberta_tokenizer.convert_tokens_to_ids(chunk)
                input_ids = torch.tensor(token_ids).unsqueeze(0).to(device)
                with torch.no_grad():
                    output = roberta_model(input_ids)
                chunk_outputs.append(output.last_hidden_state[0].cpu())
            token_embeddings = torch.cat(chunk_outputs, dim=0)
            
            edge0, edge1 = [], []
            for word in doc:
                edge0.append(word.i)
                edge1.append(word.head.i)
            edge_index = torch.tensor([edge0, edge1], dtype=torch.long)
            
            all_token_embeddings.append(token_embeddings)
            all_edge_index.append(edge_index)
            valid_labels.append(labels[idx])
        except Exception as e:
            errors += 1
    
    y = torch.tensor(valid_labels, dtype=torch.float32)
    print(f"  OK: {len(valid_labels)}, Errors: {errors}")
    return {'all_token_embeddings': all_token_embeddings, 'all_edge_index': all_edge_index, 'y': y}

In [ ]:
# Build đồ thị — khoảng 10-20 phút trên Kaggle GPU
start = time.time()

print("[1/3] Building TRAIN graphs...")
hc3_train_g = build_graph_data(train_texts, train_labels, "Train")

print("\n[2/3] Building VAL graphs...")
hc3_val_g = build_graph_data(val_texts, val_labels, "Val")

print("\n[3/3] Building TEST graphs...")
hc3_test_g = build_graph_data(test_texts, test_labels, "Test")

print(f"\n✅ Tổng thời gian: {(time.time()-start)/60:.1f} phút")

## Bước 4: Định nghĩa Models

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATv2Conv
from torch_geometric.data import Data

# ===== GCN2: Baseline (PRDetect gốc) =====
class GCN2(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN2, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
        self.fc = nn.Linear(output_dim, 1)
        self.dropout = nn.Dropout(0.5)
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.dropout(self.conv1(x, edge_index)))
        x = F.relu(self.dropout(self.conv2(x, edge_index)))
        x = self.fc(x)
        x = torch.mean(x, dim=0, keepdim=True)
        return torch.sigmoid(x)

# ===== GATv2: Cải tiến Phần 4 =====
class GATv2Model(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, heads=4, dropout=0.5):
        super(GATv2Model, self).__init__()
        self.conv1 = GATv2Conv(input_dim, hidden_dim, heads=heads, 
                               dropout=dropout, concat=True, add_self_loops=True)
        self.conv2 = GATv2Conv(hidden_dim * heads, output_dim, heads=1,
                               dropout=dropout, concat=False, add_self_loops=True)
        self.fc = nn.Linear(output_dim, 1)
        self.dropout = nn.Dropout(dropout)
    def forward(self, data, return_attention=False):
        x, edge_index = data.x, data.edge_index
        if return_attention:
            x, (_, alpha1) = self.conv1(x, edge_index, return_attention_weights=True)
        else:
            x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = self.dropout(x)
        if return_attention:
            x, (_, alpha2) = self.conv2(x, edge_index, return_attention_weights=True)
        else:
            x = self.conv2(x, edge_index)
        x = F.elu(x)
        x = self.dropout(x)
        x = self.fc(x)
        x = torch.mean(x, dim=0, keepdim=True)
        if return_attention:
            return torch.sigmoid(x), (alpha1, alpha2)
        return torch.sigmoid(x)

# Thống kê tham số
gcn = GCN2(768, 256, 64)
gatv2 = GATv2Model(768, 64, 64, heads=4)
print(f"GCN2  params: {sum(p.numel() for p in gcn.parameters()):,}")
print(f"GATv2 params: {sum(p.numel() for p in gatv2.parameters()):,}")

## Bước 5: Hàm Train & Evaluate

In [ ]:
import torch.optim as optim
from sklearn.metrics import roc_auc_score, f1_score

def train_model(model, train_data, val_data, model_name, epochs=30, lr=0.0005, patience=5):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(dev)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    criterion = nn.BCELoss()
    train_len = len(train_data['y'])
    val_len = len(val_data['y'])
    best_val_acc = -1
    pat_cnt = 0
    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    
    print(f"\n{'='*55}")
    print(f"  Training {model_name} | train={train_len}, val={val_len}, device={dev}")
    print(f"{'='*55}")
    
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        e_loss, correct = 0.0, 0
        for i in tqdm(range(train_len), desc=f"[{model_name}] Ep{epoch+1} Train", leave=False):
            data = Data(x=train_data['all_token_embeddings'][i],
                       edge_index=train_data['all_edge_index'][i],
                       y=train_data['y'][i]).to(dev)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y.float().view(-1,1))
            loss.backward()
            optimizer.step()
            e_loss += loss.item()
            correct += ((out>=0.5).long() == data.y.view(-1,1)).sum().item()
        tr_loss = e_loss/train_len
        tr_acc = correct/train_len
        
        model.eval()
        e_loss, correct = 0.0, 0
        with torch.no_grad():
            for i in range(val_len):
                data = Data(x=val_data['all_token_embeddings'][i],
                           edge_index=val_data['all_edge_index'][i],
                           y=val_data['y'][i]).to(dev)
                out = model(data)
                loss = criterion(out, data.y.float().view(-1,1))
                e_loss += loss.item()
                correct += ((out>=0.5).long() == data.y.view(-1,1)).sum().item()
        v_loss = e_loss/val_len
        v_acc = correct/val_len
        
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(v_acc)
        
        tag = ""
        if v_acc >= best_val_acc:
            best_val_acc = v_acc
            pat_cnt = 0
            torch.save(model.state_dict(), f'/kaggle/working/best_{model_name}.pth')
            tag = " ✓ BEST"
        else:
            pat_cnt += 1
            tag = f" ({pat_cnt}/{patience})"
        
        print(f"  Ep{epoch+1:2d}: tr_acc={tr_acc:.4f} v_acc={v_acc:.4f} "
              f"tr_loss={tr_loss:.4f} v_loss={v_loss:.4f}{tag}")
        if pat_cnt >= patience:
            print(f"  → Early stopping!")
            break
    
    print(f"  Time: {time.time()-t0:.0f}s | Best val_acc: {best_val_acc:.4f}")
    model.load_state_dict(torch.load(f'/kaggle/working/best_{model_name}.pth'))
    return model, history


def evaluate_model(model, test_data, model_name):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(dev)
    model.eval()
    test_len = len(test_data['y'])
    preds_prob = []
    correct = 0
    t0 = time.time()
    with torch.no_grad():
        for i in range(test_len):
            data = Data(x=test_data['all_token_embeddings'][i],
                       edge_index=test_data['all_edge_index'][i],
                       y=test_data['y'][i]).to(dev)
            out = model(data)
            preds_prob.append(out.item())
            correct += ((out>=0.5).long() == data.y.view(-1,1)).sum().item()
    
    y_pred = [1 if p>=0.5 else 0 for p in preds_prob]
    y_true = test_data['y'].numpy().tolist()
    acc = correct/test_len
    auc = roc_auc_score(y_true, preds_prob)
    f1 = f1_score(y_true, y_pred)
    elapsed = time.time()-t0
    print(f"  [{model_name}] Acc={acc:.4f} AUC={auc:.4f} F1={f1:.4f} Time={elapsed:.1f}s")
    return {'model': model_name, 'acc': acc, 'auc': auc, 'f1': f1, 'time': elapsed}

print("✅ Ready!")

## Bước 6: Train GCN2 (Baseline)

In [ ]:
torch.manual_seed(2024)
torch.cuda.manual_seed_all(2024)

gcn_model = GCN2(768, 256, 64)
gcn_model, gcn_hist = train_model(gcn_model, hc3_train_g, hc3_val_g, "GCN2", epochs=30, lr=0.0001, patience=5)

## Bước 7: Train GATv2 (Cải tiến)

In [ ]:
torch.manual_seed(2024)
torch.cuda.manual_seed_all(2024)

gatv2_model = GATv2Model(768, 64, 64, heads=4, dropout=0.5)
gatv2_model, gatv2_hist = train_model(gatv2_model, hc3_train_g, hc3_val_g, "GATv2", epochs=30, lr=0.0005, patience=5)

## Bước 8: So sánh trên HC3 Test

In [ ]:
print("\n" + "=" * 55)
print("  KẾT QUẢ TRÊN HC3 TEST")
print("=" * 55)
r_gcn = evaluate_model(gcn_model, hc3_test_g, "GCN2")
r_gatv2 = evaluate_model(gatv2_model, hc3_test_g, "GATv2")

print(f"\n  {'Metric':<8} {'GCN2':>8} {'GATv2':>8} {'Diff':>8}")
print("  " + "-"*36)
for m in ['acc','auc','f1']:
    d = r_gatv2[m]-r_gcn[m]
    print(f"  {m.upper():<8} {r_gcn[m]:>8.4f} {r_gatv2[m]:>8.4f} {d:>+7.4f} {'↑' if d>0 else '↓'}")

## Bước 9: Biểu đồ Learning Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(gcn_hist['train_acc'], 'b--', alpha=0.6, label='GCN2 Train')
axes[0].plot(gcn_hist['val_acc'], 'b-', lw=2, label='GCN2 Val')
axes[0].plot(gatv2_hist['train_acc'], 'r--', alpha=0.6, label='GATv2 Train')
axes[0].plot(gatv2_hist['val_acc'], 'r-', lw=2, label='GATv2 Val')
axes[0].set_title('Accuracy', fontsize=14)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(gcn_hist['train_loss'], 'b--', alpha=0.6, label='GCN2 Train')
axes[1].plot(gcn_hist['val_loss'], 'b-', lw=2, label='GCN2 Val')
axes[1].plot(gatv2_hist['train_loss'], 'r--', alpha=0.6, label='GATv2 Train')
axes[1].plot(gatv2_hist['val_loss'], 'r-', lw=2, label='GATv2 Val')
axes[1].set_title('Loss', fontsize=14)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('GCN2 vs GATv2 — Learning Curves', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/learning_curves.png', dpi=150)
plt.show()

## Bước 10: Test trên RAID (Cross-dataset)

In [ ]:
# Tải mẫu từ RAID dataset qua HuggingFace streaming (nhanh, nhẹ, không tốn RAM/disk)
print("Đang lấy mẫu từ RAID dataset (HuggingFace streaming)...")
try:
    from datasets import load_dataset
    import pandas as pd
    raid_stream = load_dataset('liamdugan/raid', split='train', streaming=True)
    
    human_samples = []
    machine_samples = []
    
    for row in raid_stream:
        gen = row.get('generation', '').strip()
        if not gen or len(gen) < 30:
            continue
        if row.get('model') == 'human' and len(human_samples) < 300:
            human_samples.append(gen)
        elif row.get('model') != 'human' and len(machine_samples) < 300:
            machine_samples.append(gen)
        if len(human_samples) >= 300 and len(machine_samples) >= 300:
            break
            
    raid_texts = human_samples + machine_samples
    raid_labels = [1]*len(human_samples) + [0]*len(machine_samples)
    combined_raid = list(zip(raid_texts, raid_labels))
    random.shuffle(combined_raid)
    raid_texts, raid_labels = zip(*combined_raid)
    raid_texts, raid_labels = list(raid_texts), list(raid_labels)
    
    print(f'✅ Đã lấy thành công {len(raid_texts)} mẫu RAID (300 Human, 300 AI) qua streaming!')
    RAID_LOADED = True
except Exception as e:
    print(f'⚠️ Không tải được RAID: {e}')
    RAID_LOADED = False


In [ ]:
if RAID_LOADED:
    print(f'RAID subset: {len(raid_texts)} mẫu')
    
    print('\nBuilding RAID graphs...')
    raid_test_g = build_graph_data(raid_texts, raid_labels, 'RAID')
    
    print('\n' + '='*55)
    print('  KẾT QUẢ TRÊN RAID (Cross-dataset)')
    print('='*55)
    rr_gcn = evaluate_model(gcn_model, raid_test_g, 'GCN2')
    rr_gatv2 = evaluate_model(gatv2_model, raid_test_g, 'GATv2')
    
    print(f'\n  {"Metric":<8} {"GCN2":>8} {"GATv2":>8} {"Diff":>8}')
    print('  ' + '-'*36)
    for m in ['acc', 'auc', 'f1']:
        d = rr_gatv2[m] - rr_gcn[m]
        print(f'  {m.upper():<8} {rr_gcn[m]:>8.4f} {rr_gatv2[m]:>8.4f} {d:>+7.4f} {"↑" if d>0 else "↓"}')
else:
    print('Bỏ qua RAID test (dataset chưa load được)')


## Bước 11: Bảng tổng hợp & Lưu kết quả

In [ ]:
import pandas as pd

results = [
    {**r_gcn, 'dataset': 'HC3'},
    {**r_gatv2, 'dataset': 'HC3'}
]
if RAID_LOADED:
    results.append({**rr_gcn, 'dataset': 'RAID'})
    results.append({**rr_gatv2, 'dataset': 'RAID'})

df = pd.DataFrame(results)[['dataset','model','acc','auc','f1','time']]
df.columns = ['Dataset','Model','Accuracy','AUC-ROC','F1','Inference(s)']

print("\n" + "="*65)
print("  📊 TỔNG HỢP: GCN2 (Baseline) vs GATv2 (Cải tiến Phần 4)")
print("="*65)
print(df.to_string(index=False, float_format='%.4f'))
print("="*65)

df.to_csv('/kaggle/working/results.csv', index=False)
print("\n💾 Saved: /kaggle/working/results.csv")
print("💾 Models: /kaggle/working/best_GCN2.pth, best_GATv2.pth")
print("💾 Chart: /kaggle/working/learning_curves.png")
print("\n✅ HOÀN TẤT! Tải kết quả từ tab Output bên phải.")